# TENTACLES: D1 discovery preprocessing

Official TENTACLES v0.1.0 at pinned Git revision. Source cohort: D1 (PRJNA248469), 260 samples. This run saves the original package's preprocessing output and documents the `runClassifiers()` API. It does not train classifiers or reproduce the final paper signature.

In [1]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import os
import platform
import shutil
import subprocess
import tarfile
import tempfile

try:
    from google.colab import drive
    drive.mount("/content/drive")
except ModuleNotFoundError:
    pass

candidates = [Path("/content/drive/MyDrive/CMPS355_Research"), Path.cwd(), *Path.cwd().parents]
ROOT = next((p for p in candidates if (p / "configs" / "dataset.json").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("CMPS355_Research project not found")
CONFIG = json.loads((ROOT / "configs" / "dataset.json").read_text(encoding="utf-8"))
DATA = ROOT / CONFIG["artifact"]
SCRIPT = ROOT / "r" / "tentacles_d1_preprocess.R"
if not DATA.is_file() or not SCRIPT.is_file():
    raise FileNotFoundError(f"Required file missing: data={DATA.is_file()}, R script={SCRIPT.is_file()}")
with DATA.open("rb") as f:
    digest = hashlib.file_digest(f, "sha256").hexdigest()
if digest != CONFIG["sha256"]:
    raise RuntimeError("Accepted dataset SHA-256 does not match the manifest")
RSCRIPT = shutil.which("Rscript")
if RSCRIPT is None:
    raise RuntimeError("Rscript is unavailable")
SOURCE_SHA = "069a26271e3885e015df25baa40a7a1140827d5f"
LIB = Path("/content/cmps355_r_lib")
RUNS = ROOT / "runs" / "tentacles_d1"
RUNS.mkdir(parents=True, exist_ok=True)
print("Dataset checksum verified:", digest)
print("Rscript:", RSCRIPT)
print("Project:", ROOT)

Mounted at /content/drive
Dataset checksum verified: 8a83f20035fcac5ae5d8bfe496e4d7491e47400b61ca1080b828c8ad54e91a74
Rscript: /usr/bin/Rscript
Project: /content/drive/MyDrive/CMPS355_Research


## Verified R environment

In [2]:
r_version = subprocess.check_output(
    [RSCRIPT, "--vanilla", "-e", "cat(as.character(getRversion()))"], text=True
).strip()
os_info = platform.freedesktop_os_release()
key = f"r{r_version}_{os_info.get('ID','linux')}{os_info.get('VERSION_ID','unknown')}_{platform.machine()}"
cache_dir = ROOT / "runs" / "environment_cache"
archive = cache_dir / f"{key}.tar.gz"
manifest_file = cache_dir / f"{key}.json"

def installed_revision():
    env = os.environ.copy()
    env["CMPS355_R_LIB"] = str(LIB)
    expr = (
        'p <- Sys.getenv("CMPS355_R_LIB"); '
        'if (dir.exists(p)) .libPaths(c(p,.libPaths())); '
        'if (!requireNamespace("TENTACLES", quietly=TRUE)) quit(status=2L); '
        'cat(tolower(utils::packageDescription("TENTACLES")[["RemoteSha"]]))'
    )
    result = subprocess.run(
        [RSCRIPT, "--vanilla", "-e", expr],
        env=env, capture_output=True, text=True
    )
    return result.stdout.strip() if result.returncode == 0 else None

revision = installed_revision()
if revision != SOURCE_SHA:
    if not archive.is_file() or not manifest_file.is_file():
        raise RuntimeError(
            f"Official TENTACLES {SOURCE_SHA} unavailable and no compatible cache found: {archive}. "
            "Restore the environment using notebook 04 before running this analysis."
        )
    manifest = json.loads(manifest_file.read_text(encoding="utf-8"))
    expected = {
        "r_version": r_version,
        "os": os_info.get("ID", "linux"),
        "os_version": os_info.get("VERSION_ID", "unknown"),
        "arch": platform.machine()
    }
    if any(manifest.get(k) != v for k, v in expected.items()):
        raise RuntimeError("R cache environment is incompatible with this runtime")
    with archive.open("rb") as f:
        if hashlib.file_digest(f, "sha256").hexdigest() != manifest.get("sha256"):
            raise RuntimeError("R package cache checksum mismatch")
    LIB.mkdir(parents=True, exist_ok=True)
    with tempfile.TemporaryDirectory() as temp:
        with tarfile.open(archive, "r:gz") as tf:
            for entry in tf.getmembers():
                parts = Path(entry.name).parts
                if not parts or parts[0] != "r_library" or ".." in parts:
                    raise RuntimeError("Invalid R cache archive layout")
            tf.extractall(temp, filter="data")
        src = Path(temp) / "r_library"
        if not src.is_dir():
            raise RuntimeError("R cache does not contain r_library")
        for package in src.iterdir():
            if package.is_dir() and not (LIB / package.name).exists():
                shutil.copytree(package, LIB / package.name)
    revision = installed_revision()
if revision != SOURCE_SHA:
    raise RuntimeError(f"Installed package revision does not match expected source: {revision}")
print("R:", r_version)
print("TENTACLES source:", revision)

R: 4.6.1
TENTACLES source: 069a26271e3885e015df25baa40a7a1140827d5f


## D1 preprocessing with the original implementation

In [3]:
stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
run_dir = RUNS / stamp
run_dir.mkdir(parents=True, exist_ok=False)
log_file = run_dir / "preprocess.log"
env = os.environ.copy()
env["CMPS355_R_LIB"] = str(LIB)
command = [RSCRIPT, "--vanilla", str(SCRIPT), str(DATA), str(run_dir), SOURCE_SHA, digest]
with log_file.open("w", encoding="utf-8") as stream:
    try:
        result = subprocess.run(command, stdout=stream, stderr=subprocess.STDOUT, env=env, timeout=1800)
        exit_code = result.returncode
    except subprocess.TimeoutExpired:
        exit_code = 124
        stream.write("\nTIMEOUT_EXCEEDED\n")
lines = log_file.read_text(encoding="utf-8", errors="replace").splitlines()
required = {
    "SOURCE_SHA=" + SOURCE_SHA,
    "SAMPLE_ALIGNMENT=PASS",
    "INPUT_SAMPLES=260",
    "CLASS_CD=218",
    "CLASS_NOT_IBD=42",
    "PREPROCESS_EXECUTED=TRUE",
    "RUNCLASSIFIERS_API_CAPTURED=TRUE",
    "STATUS=PASS_D1_OFFICIAL_PREPROCESS"
}
passed = exit_code == 0 and required.issubset(set(lines)) and (run_dir / "d1_preprocessed.rds").is_file()
report = {
    "method": "TENTACLES", "stage": "D1_discovery_preprocessing",
    "status": "PASS_D1_PREPROCESS" if passed else "FAIL_D1_PREPROCESS",
    "source_commit": SOURCE_SHA,
    "dataset_sha256": digest,
    "r_script_sha256": hashlib.sha256(SCRIPT.read_bytes()).hexdigest(),
    "exit_code": exit_code,
    "paper_track": "D1 only; classifier execution not started",
    "files": {},
    "created_utc": datetime.now(timezone.utc).isoformat()
}
for path in sorted(run_dir.iterdir()):
    if path.is_file() and path.name != "report.json":
        with path.open("rb") as f:
            report["files"][path.name] = {
                "sha256": hashlib.file_digest(f, "sha256").hexdigest(),
                "bytes": path.stat().st_size
            }
report_file = run_dir / "report.json"
report_file.write_text(json.dumps(report, indent=2) + "\n", encoding="utf-8")
print("\n".join(lines[-30:]))
print("Report:", report_file.relative_to(ROOT))
print("Status:", report["status"])
if not passed:
    raise RuntimeError("Official D1 preprocessing failed; inspect preprocess.log in the saved run folder")

METHOD=TENTACLES
SCOPE=D1_ONLY_PUBLISHED_DISCOVERY_PREPROCESSING
SOURCE_SHA=069a26271e3885e015df25baa40a7a1140827d5f
PACKAGE_VERSION=0.1.0
R_VERSION=4.6.1
EXPECTED_DATA_SHA256=8a83f20035fcac5ae5d8bfe496e4d7491e47400b61ca1080b828c8ad54e91a74
INPUT_SAMPLES=260
INPUT_GENES=10061
CLASS_CD=218
CLASS_NOT_IBD=42
SAMPLE_ALIGNMENT=PASS
PREPROCESS_PARAMETERS=data.type:rnaseq;is.normalized:FALSE;mincpm:1;minfraction:0.1;batch:NULL;covar.mod:NULL;plot:FALSE

── preProcess ──────────────────────────────────────────────────────────────────
ℹ Importing data...
ℹ Transforming class labels to binary factors...
ℹ Transforming 'Not IBD' to 0 and 'CD' to 1.
✔ Data Imported!
ℹ Normalizing and filtering data...
✔ Data Normalized!
✔ Data pre-processing complete!
PREPROCESS_EXECUTED=TRUE
PREPROCESS_RESULT_CLASS=preProcess.obj
PREPROCESS_RESULT_NAMES=
PREPROCESS_RDS_BYTES=36333007
AUTHOR_RUNCLASSIFIERS_DOCUMENTATION=AVAILABLE
RUNCLASSIFIERS_API_CAPTURED=TRUE
STATUS=PASS_D1_OFFICIAL_PREPROCESS
Report: runs/tent